In [1]:
!apt-get update && apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip install faiss-cpu pandas numpy requests

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,701 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,090 kB]
Hit:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:13 http://security.ubuntu.com/ubuntu jammy-security/u

In [2]:
import subprocess
import time
import requests
import pandas as pd
import numpy as np
import faiss
import json

# Start Ollama
print("Starting Ollama server...")
# Ensure any previous Ollama process is stopped or restart the service
# subprocess.run(['killall', 'ollama'], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Wait for server to be up
max_retries = 30 # Increased retries for robustness
server_up = False
for i in range(max_retries):
    try:
        response = requests.get("http://localhost:11434", timeout=1) # Added timeout
        if response.status_code == 200:
            print("Ollama server is up!")
            server_up = True
            break
    except requests.exceptions.ConnectionError:
        print(f"Waiting for Ollama server... ({i+1}/{max_retries})")
        time.sleep(2)
    except Exception as e:
        print(f"An unexpected error occurred while checking Ollama server: {e}")
        time.sleep(2)

if not server_up:
    raise Exception("Failed to start Ollama server after multiple retries.")

# Pull Phi3 model after server is confirmed to be up
print("Pulling Phi3 model...")
# subprocess.run is blocking, which is good here to ensure model is pulled before next steps.
pull_command = ['ollama', 'pull', 'phi3']
pull_process = subprocess.run(pull_command, capture_output=True, text=True)

if pull_process.returncode != 0:
    print(f"Error pulling Phi3 model:\n{pull_process.stderr}")
    raise Exception(f"Failed to pull Phi3 model: {pull_process.stderr}")
else:
    print("Phi3 model pulled successfully.")
    print(pull_process.stdout)

# Give Ollama a little more time to load the model after pulling
print("Giving Ollama a moment to load the model...")
time.sleep(10) # Increased sleep time for model loading

Starting Ollama server...
Waiting for Ollama server... (1/30)
Ollama server is up!
Pulling Phi3 model...
Phi3 model pulled successfully.

Giving Ollama a moment to load the model...


In [3]:
def ollama_embed(text):
    res = requests.post("http://localhost:11434/api/embeddings",
                        json={"model": "phi3", "prompt": text})
    return res.json()['embedding']

# Load Data
df = pd.read_csv('/content/sample_data/california_housing_train.csv')
subset = df.head(100) # Using a subset for speed

documents = []
for _, row in subset.iterrows():
    doc = f"Location: ({row['latitude']}, {row['longitude']}). Age: {row['housing_median_age']}, Rooms: {row['total_rooms']}, Income: {row['median_income']}, Value: {row['median_house_value']}"
    documents.append(doc)

print("Generating local embeddings...")
embeddings = np.array([ollama_embed(d) for d in documents]).astype('float32')

# Build Index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)
print(f"Index ready with {index.ntotal} documents (Dimension: {dimension})")

Generating local embeddings...
Index ready with 100 documents (Dimension: 3072)


In [5]:
def local_rag_query(query):
    # Retrieve
    q_emb = np.array([ollama_embed(query)]).astype('float32')
    _, indices = index.search(q_emb, k=3)
    context = "\n".join([documents[i] for i in indices[0]])

    # Generate
    prompt = f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"
    res = requests.post("http://localhost:11434/api/generate",
                        json={"model": "phi3", "prompt": prompt, "stream": False})
    return res.json()['response']

# @title Ask a Question
user_query = "What is oracle?" # @param {type:"string"}
print(local_rag_query(user_query))

The term "oracle" does not directly relate to the given context about real estate data with various locations, ages of properties, number of rooms, income levels, and property values in specific coordinates. An Oracle typically refers to a powerful computer program or system used for processing large amounts of information rapidly (in computing) or can also refer to an ancient Greek deity through whom divine wisdom was imparted (in mythology). Since the question is unrelated to real estate data analysis, it seems like you might be asking about something different. Could you please provide more context on what "oracle" refers to in your query?
